# Virus spread model 

## Model Overview  

The development of the model was guided by the central research question concerning the efficiency of different travel restrictions. The model programmed using python utilizing the mesa framework is designed to simulate the international spread of a hypothetical pathogen transmitted via agents traveling across the borders of fictional micro-countries. The utility of the model lies in its multi-level architecture, upscaling the micro-dynamics of citizen agents (infection, immunity and travel propensity) interacting within the environmental agents (countries) that regulate the simulation landscape, revealing the macro-dynamics between citizen behavior and state-imposed border policies. This framework ultimately aims to determine which travel restriction strategy most efficiently mitigates viral transmission. 

## Agent Architecture 

The model contains six environmental country agents which are each assigned one distinct boarder control strategy and monitor their internal system state (aggregate infection levels). Their goal is to maintain a "Healthy" system state by filtering incoming Mobile Agents based on their assigned policy.  

Citizen agents possess internal states (Health Status: Susceptible, Infected, Immune) and stochastic behaviors (travel propensity). They act as the carriers of information (the virus) between countries. 

Countries are environmental agents which contain citizens; citizens are reactionary agents which simulate the infection process and participate in cross boarder travel. The model assumes a complete graph for international travel, meaning a citizen can travel from their home country to any other country, provided they satisfy the border restrictions of both nations.  

Every simulation run initializes each country with a population of 1,000 citizens, while the location of the initial viral seed varies across iterations. The main parameters adjusted between iterations are the travel rate (travel propensity) and the initial seed country. The simulation proceeds in discrete time steps, where one step represents one day.  

The system dynamics are driven by the decision cycle of the citizen agents, which forces interactions with the country agents. The agent activity per time step can be summed up in three main stages:  

The agent updates its infection time, changing its status to immune upon reaching the recovery time when infected, or upon reaching the immunity time when immune. If it is neither infected, nor immune, it tracks how long it has been healthy for; this metric is use for quarantine measures.  

The agent meets others in its current country, with a 5% probability of transmitting its potential infection. 

Given a specified travel-rate parameter, the citizen agent travels to a randomly rolled other country for a defined duration. Upon deciding to travel, an agent confirms that it is allowed to travel by both the current country’s and the destination country’s border policies; if it cannot, the destination is saved for the next travel attempt.  

After all the agents have acted, the time step finishes by updating the metrics. 

The model assumes a complete graph for international travel, meaning a citizen can travel from their home country to any other country, provided they satisfy the border restrictions of both nations.  

### Importing necessary libraries and declaring global variables

In [ ]:
import mesa
from mesa import Agent, Model
from mesa.datacollection import DataCollector
import pandas as pd
from mesa.datacollection import DataCollector
from mesa.visualization import SolaraViz, make_plot_component
from scipy.signal import find_peaks

STRATEGY_LABELS = [
    "Open Borders", "Short Quarantine", "Long Quarantine",
    "Negative Test", "VIP Only", "Full Lockdown"
]
NUM_STRATEGIES = 6
COUNTRIES_CONFIG = [
    {'name': 'A', 'strategy': 0, 'population': 1000},
    {'name': 'B', 'strategy': 1, 'population': 1000},
    {'name': 'C', 'strategy': 2, 'population': 1000},
    {'name': 'D', 'strategy': 3, 'population': 1000},
    {'name': 'E', 'strategy': 4, 'population': 1000},
    {'name': 'F', 'strategy': 5, 'population': 1000},
]

### Citizen Agent setup

Citizens are reactionary agents which simulate the infection process and participate in cross boarder travel. Every day, the activities of every citizen agent can be summarized as three major steps: 

1. The agent updates its infection time, changing its status to immune upon reaching the recovery time when infected, or upon reaching the immunity time when immune. If it is neither infected, nor immune, it tracks how long it has been healthy for; this metric is use for quarantine measures.  

2. The agent meets others in its current country, with a certain probability of transmitting its potential infection. 

3. Given a specified travel-rate parameter, the citizen agent travels to a randomly rolled other country for a defined duration. Upon deciding to travel, an agent confirms that it is allowed to travel by both the current country’s and the destination country’s border policies; if it cannot, the destination is saved for the next travel attempt.  

In [2]:
class Citizen(Agent):
    def __init__(self, unique_id, model, home_country):
        super().__init__(model)
        # set default values
        self.unique_id = unique_id

        # position values 
        self.home_country = home_country
        self.current_country = home_country

        # Infection values 
        self.infected = False
        self.immune = False
        self.days_infected = 0 # days for citizen spent infected
        self.days_immune = 0 # days for citizen spent immune
        self.recovery_days = 0 # days for citizen to recover
        self.immunity_days = 0 # days for citizen to stay immune
        self.days_since_last_infection = 0 # citizen's days since last infection

        # Values for travelling
        self.travel_days = 0
        self.destination = None
        self.is_vip = self.model.rng.random() <= self.model.vip_rate


    def can_travel(self, destination):
        strat = destination.strategy
        if strat == 0:  # Open borders
            return True
        elif strat == 1:  # Short quarantine
            return self.days_since_last_infection >= self.model.short_quarantine_time
        elif strat == 2:  # Long quarantine
            return self.days_since_last_infection >= self.model.long_quarantine_time
        elif strat == 3:  # Negative test
            # Test may be wrong, but travel logic should allow attempts
            if self.infected:
                return self.model.rng.random() < self.model.test_wrong_rate
            else:
                return self.model.rng.random() > self.model.test_wrong_rate
        elif strat == 4:  # VIP only
            return self.is_vip
        elif strat == 5:  # Full lockdown
            return False
        return False

    def meet_people(self, citizens_met): 
        for citizen in citizens_met:
            if self.infected and not citizen.immune and not citizen.infected: # Only infect if the other citizen is not immune or already infected
                if self.model.rng.random() <= self.model.infection_rate:
                    # Infection of another citizen
                    citizen.infected = True
                    citizen.recovery_days = self.model.rng.choice(self.model.recovery_distr, p=self.model.recovery_distr_weights)
                    citizen.immunity_days = 2 * citizen.recovery_days

    def step(self):
        # 1. Infection status update
        if self.infected:
            self.days_infected += 1
            if self.days_infected == self.recovery_days:
                self.infected = False
                self.immune = True
                self.days_infected = 0
                self.days_since_last_infection = 0
                self.days_immune = 0
        else:
            self.days_since_last_infection += 1

        if self.immune:
            self.days_immune += 1
            if self.days_immune == self.immunity_days:
                self.immune = False

        # 2. Meet people locally 
        others = [c for c in self.current_country.people if c is not self]
        n_meet = min(self.model.daily_meetings, len(others))
        if n_meet > 0 and self.infected:
            people_met = self.model.rng.choice(others, n_meet, replace=False)
            self.meet_people(people_met)

        # 3. Travel decision (AFTER meetings; keeps infection local)
        if self.home_country.unique_id == self.current_country.unique_id:  # Only consider travel if in home country
            if self.model.rng.random() < self.model.travel_rate:
                potential = [c for c in self.model.country_agents if c is not self.current_country]
                if potential:
                    if self.destination is None:
                        self.destination = self.model.rng.choice(potential)
                    if self.can_travel(self.destination) and self.can_travel(self.current_country):
                        # Remove from old country
                        self.current_country.people.remove(self)
                        # Move to new country
                        self.current_country = self.destination
                        self.current_country.people.append(self)
                        self.travel_days = self.model.rng.integers(self.model.travel_days_min, self.model.travel_days_max+1)
                        self.destination = None
        else:
            self.travel_days -= 1
            if self.travel_days <= 0:
                # Return home
                self.current_country.people.remove(self)
                self.current_country = self.home_country
                self.home_country.people.append(self)

### Country Agent setup

Countries are environmental agents which contain citizens

In [3]:
class Country(Agent):
    def __init__(self, name, unique_id, model, strategy, population):
        super().__init__(model)
        # set default values for country
        self.unique_id = unique_id
        self.name = name
        self.strategy = strategy
        self.population = population
        self.people = []

    def step(self):
        pass

### Virus spread (Model) setup

* Instances of countries and their citizens are created
* Model step function implementation
* Data collector for the batch runner (later needed for the final csv files)

In [ ]:
class VirusSpreadModel(Model):
    def __init__(self,
                    vip_rate = 0.05,
                    daily_meetings = 10, 
                    infection_rate = 0.05,
                    travel_rate = 1/365,
                    travel_days_min = 7,
                    travel_days_max = 14,
                    short_quarantine_time = 7,
                    long_quarantine_time = 14,
                    recovery_distr = [7, 8, 9, 10, 11, 12, 13, 14],
                    recovery_distr_weights = [0.05, 0.05, 0.15, 0.25, 0.25, 0.15, 0.05, 0.05],
                    test_wrong_rate = 0.2,
                    countries_config = COUNTRIES_CONFIG, 
                    seed_country = None,
                    seed = None
        ):
        super().__init__(seed=seed)
        # set default values for the model (parameters)
        self.steps = 0
        self.infection_rate = infection_rate
        self.daily_meetings = daily_meetings
        self.vip_rate = vip_rate
        self.travel_rate = travel_rate
        self.travel_days_min = travel_days_min
        self.travel_days_max = travel_days_max
        self.short_quarantine_time = short_quarantine_time
        self.long_quarantine_time = long_quarantine_time
        self.recovery_distr = recovery_distr
        self.recovery_distr_weights = recovery_distr_weights
        self.test_wrong_rate = test_wrong_rate
        self.seed_country = seed_country
        self.country_agents = []

        # Creating countries 
        for conf in countries_config:
            country = Country(conf["name"], conf["name"], self, conf["strategy"], conf["population"])
            self.country_agents.append(country)

        # Function to calculate infection percent per strategy
        def infection_percent_for_strategy(target_strategy):
            def _calc_infection(model):
                infected_total = 0
                total_people = 0   

                # Loop through every country
                for ctry in model.country_agents:
                    if ctry.strategy == target_strategy:
                        total_people += len(ctry.people)
                        for person in ctry.people:
                            if person.infected:
                                infected_total += 1
                                
                # handle edge case of no people in strategy
                if total_people == 0:
                    return 0.0
                
                pct = infected_total / total_people
                if pct < 0:
                    pct = 0.0
                elif pct > 1:
                    pct = 1.0

                return pct
            return _calc_infection

        # Data Collector setup
        self.datacollector = DataCollector(
            model_reporters={
                f"{label}_InfectionPercent": infection_percent_for_strategy(idx)
                for idx, label in enumerate(STRATEGY_LABELS)
            }

        )

        
        # Create citizens
        for country in self.country_agents:
            for i in range(country.population):
                citizen = Citizen(unique_id=f"{country.name}_{i}", model=self, home_country = country)
                country.people.append(citizen)
            if country.strategy == self.seed_country:
                country.people[0].infected = True
                country.people[0].recovery_days = self.rng.choice(self.recovery_distr, p = self.recovery_distr_weights)
                country.people[0].immunity_days = 2 * country.people[0].recovery_days
        
    def step(self):
        self.datacollector.collect(self)
        for country in self.country_agents:
            for citizen in country.people:
                citizen.step()


### Solaraviz visualisation for adjusting parameters 

We used mesa visualisation module to run our model under different parameters. 

In [5]:
model_params = {
    "travel_rate": {
    "type": "SliderFloat",
    "value": 1/365, 
    "label": "Travel rate",
    "min": 0.0, 
    "max": 0.1, 
    "step": 0.001
    },
    "seed_country": {
    "type": "SliderInt",
    "value": 0,
    "label": "Seed Country (Strategy Index)",
    "min": 0,
    "max": 5,
    "step": 1
    }
}

In [6]:
model = VirusSpreadModel()
page = SolaraViz(
    model,
    model_params=model_params,
    components=[
        make_plot_component([f"{label}_InfectionPercent" for label in STRATEGY_LABELS])
    ]
)

In [7]:
page

Cannot show ipywidgets in text

### Experiment 1: Generating first csv file with 6 different strategy countries together

In the first experiment, we ran our model with six different border strategies (excluding closed borders), systematically varying the infection seed country. We tested each configuration across three travel rate values: traveling once every three years, once per year, and three times per year.

First cell generates dataframe with infection spread results
Consequent cell adds encoded first peak values to the existing dataframe and saves it into a csv file, which is then used for the further analysis.

In [ ]:
# Define batch parameters as before
batch_params = {
    "seed_country": [0, 1, 2, 3, 4],
    "travel_rate": [1/1095, 1/365, 3/365]
}

# Run batch and save full output
results = mesa.batch_run(
    VirusSpreadModel,
    parameters=batch_params,
    iterations=1,
    max_steps=50,
    number_processes=1,
    data_collection_period=1,
    display_progress=True,
)
results_df = pd.DataFrame(results)

  0%|          | 0/5 [00:00<?, ?it/s]

In [9]:
# Add indicator columns
for label in STRATEGY_LABELS:
    indicator_col = f"{label}_IsFirstPeak"
    results_df[indicator_col] = 0

# Calculate first peaks for each run 
for runid in results_df['RunId'].unique():
    run_rows = results_df[results_df['RunId'] == runid]
    for label in STRATEGY_LABELS:
        col = f"{label}_InfectionPercent"
        indicator_col = f"{label}_IsFirstPeak"
        if col in run_rows:
            vals = run_rows[col].values * 100
            peaks, _ = find_peaks(vals, prominence=0.5)
            if len(peaks) > 0:
                first_peak_idx = peaks[0]
                absolute_index = run_rows.index[first_peak_idx]
                results_df.loc[absolute_index, indicator_col] = 1

# Save to a new file
results_df.to_csv("virus_spread_batch_with_firstpeak_indicator.csv", index=False)

### Experiment 2: Generating second csv file with countries pairwise comparison

In the second experiment, we ran our model in four different pairwise strategy configurations, testing each across three travel rate values: traveling once every three years, once per year, and three times per year:
1. Open Borders (infection seed country) vs Short Quarantine
2. Open Borders (infection seed country) vs Long Quarantine
3. Open Borders (infection seed country) vs Negative Test
4. Open Borders (infection seed country) vs VIP Only

First cell generates dataframe with infection spread results
Consequent cell adds encoded first peak values to the existing dataframe and saves it into a csv file, which is then used for the further analysis.

In [ ]:
# Define the 4 pairwise combinations
pairwise_configs = {
    0: [  # Open Borders vs Short Quarantine
        {'name': 'A_OpenBorders', 'strategy': 0, 'population': 1000},
        {'name': 'B_ShortQuarantine', 'strategy': 1, 'population': 1000}
    ],
    1: [  # Open Borders vs Long Quarantine
        {'name': 'A_OpenBorders', 'strategy': 0, 'population': 1000},
        {'name': 'C_LongQuarantine', 'strategy': 2, 'population': 1000}
    ],
    2: [  # Open Borders vs Negative Test
        {'name': 'A_OpenBorders', 'strategy': 0, 'population': 1000},
        {'name': 'D_NegativeTest', 'strategy': 3, 'population': 1000}
    ],
    3: [  # Open Borders vs VIP Only
        {'name': 'A_OpenBorders', 'strategy': 0, 'population': 1000},
        {'name': 'E_VIPOnly', 'strategy': 4, 'population': 1000}
    ]
}

# Batch parameters
batch_params = {
    "pair_id": [0, 1, 2, 3],  
    "seed_country": [0],  
    "travel_rate": [1/1095, 1/365, 3/365],  
}

# Wrapper function to create model for each pair
def create_model_for_pair(pair_id=0, **kwargs):
    kwargs['countries_config'] = pairwise_configs[pair_id]
    kwargs.pop('pair_id', None)
    return VirusSpreadModel(**kwargs)

# Run batch
results = mesa.batch_run(
    create_model_for_pair,
    parameters=batch_params,
    iterations=1,  
    max_steps=50,
    number_processes=1,
    data_collection_period=1,
    display_progress=True,
)

results_df = pd.DataFrame(results)

  0%|          | 0/4 [00:00<?, ?it/s]

In [10]:
# Add indicator columns
for label in STRATEGY_LABELS:
    indicator_col = f"{label}_IsFirstPeak"
    results_df[indicator_col] = 0

# Calculate first peaks for each run
for runid in results_df['RunId'].unique():
    run_rows = results_df[results_df['RunId'] == runid]
    for label in STRATEGY_LABELS:
        col = f"{label}_InfectionPercent"
        indicator_col = f"{label}_IsFirstPeak"
        if col in run_rows.columns:
            vals = run_rows[col].values * 100
            peaks, _ = find_peaks(vals, prominence=0.5)
            if len(peaks) > 0:
                first_peak_idx = peaks[0]
                absolute_index = run_rows.index[first_peak_idx]
                results_df.loc[absolute_index, indicator_col] = 1

results_df.to_csv("pairwise_batch_with_peaks.csv", index=False)